# Imports

In [59]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb

In [60]:
df = pd.read_csv('../ml/df_model.csv', sep=';', dtype={'Code_INSEE': str})
display(df.head(5))
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df['Résultat'].value_counts())
print(f"\nProportions :")
print(df['Résultat'].value_counts(normalize=True).round(4) * 100)

print('\nColonnes :')
print(df.columns)

,Code_INSEE,Résultat,Femmes,Hommes,Agriculteurs,Artisans,Cadres,Intermédiaires,Employés,Ouvriers,...,55-64 ans,65-79 ans,80 ans et +,Mariés,Pacsés,Concubinage,Veufs,Divorcés,Célibataires,Population_active
0,01001,droite,47.184259,52.113958,0.704243,5.633941,10.563640,14.789096,15.493339,17.606067,...,18.794835,18.364419,6.312769,51.219512,7.890961,16.499283,5.882353,3.156385,15.351506,697.000000
1,01002,gauche,41.363637,50.555556,11.489899,2.297980,9.191919,13.787879,9.191919,9.191919,...,12.037037,14.814815,8.333333,47.685185,11.111111,9.722222,3.240741,7.407407,20.833333,216.000000
2,01004,droite,52.881826,47.118174,0.127297,3.114317,7.806622,15.437689,17.760558,15.886919,...,14.436934,16.198957,6.106620,40.957640,5.835323,9.955739,6.778822,8.566899,27.644693,12594.508695
3,01005,droite,51.346262,47.882309,0.362612,6.556032,8.691036,15.289281,21.150485,13.928360,...,17.295346,15.384787,5.562916,48.809515,7.840915,14.193995,4.623646,4.509098,20.022831,1554.355560
4,01006,droite,39.860139,54.807692,0.000000,0.000000,0.000000,9.965035,14.947553,44.842658,...,23.076923,25.000000,5.769231,49.038462,8.653846,10.576923,2.884615,9.615385,19.230769,104.000000


Dataset chargé : 34715 lignes, 35 colonnes

Distribution de la cible (vote politique) :
Résultat
droite    26138
centre     5079
gauche     3498
Name: count, dtype: int64

Proportions :
Résultat
droite    75.29
centre    14.63
gauche    10.08
Name: proportion, dtype: float64

Colonnes :
Index(['Code_INSEE', 'Résultat', 'Femmes', 'Hommes', 'Agriculteurs',
       'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers',
       'Retraités', 'Etudiants', 'Inactifs', 'Personne seule', 'Homme seul',
       'Femme seule', 'Colocation', 'Famille', 'Famille monoparentale',
       'Couple sans enfant', 'Couple avec enfants', 'Population avec enfants',
       '15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans',
       '80 ans et +', 'Mariés', 'Pacsés', 'Concubinage', 'Veufs', 'Divorcés',
       'Célibataires', 'Population_active'],
      dtype='str')


# Feature engineering

### Ordinal encoding de la variable cible

In [61]:
groups = {'gauche': 0, 'centre': 1, 'droite': 2}
df['Résultat'] = df['Résultat'].replace(groups).astype(int)
display(df)

,Code_INSEE,Résultat,Femmes,Hommes,Agriculteurs,Artisans,Cadres,Intermédiaires,Employés,Ouvriers,...,55-64 ans,65-79 ans,80 ans et +,Mariés,Pacsés,Concubinage,Veufs,Divorcés,Célibataires,Population_active
0,01001,2,47.184259,52.113958,0.704243,5.633941,10.563640,14.789096,15.493339,17.606067,...,18.794835,18.364419,6.312769,51.219512,7.890961,16.499283,5.882353,3.156385,15.351506,697.000000
1,01002,0,41.363637,50.555556,11.489899,2.297980,9.191919,13.787879,9.191919,9.191919,...,12.037037,14.814815,8.333333,47.685185,11.111111,9.722222,3.240741,7.407407,20.833333,216.000000
2,01004,2,52.881826,47.118174,0.127297,3.114317,7.806622,15.437689,17.760558,15.886919,...,14.436934,16.198957,6.106620,40.957640,5.835323,9.955739,6.778822,8.566899,27.644693,12594.508695
3,01005,2,51.346262,47.882309,0.362612,6.556032,8.691036,15.289281,21.150485,13.928360,...,17.295346,15.384787,5.562916,48.809515,7.840915,14.193995,4.623646,4.509098,20.022831,1554.355560
4,01006,2,39.860139,54.807692,0.000000,0.000000,0.000000,9.965035,14.947553,44.842658,...,23.076923,25.000000,5.769231,49.038462,8.653846,10.576923,2.884615,9.615385,19.230769,104.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34710,95676,2,54.649893,44.135551,0.000000,5.257478,9.135020,24.653834,20.575269,6.438841,...,16.530655,16.979097,4.528522,47.392614,4.639004,15.669851,5.102841,6.176331,21.019359,384.624042
34711,95678,2,45.606723,51.307563,0.000000,7.838655,16.389916,20.665546,10.689076,5.700840,...,17.428571,13.714286,7.142857,45.714286,10.142857,12.857143,5.000000,4.571429,21.714286,700.000000
34712,95680,0,51.602206,48.397794,0.000000,2.962626,4.086016,11.996908,20.954176,17.632084,...,13.628300,10.714654,3.277626,44.969973,1.178277,7.481139,4.148062,5.286505,36.832896,21596.176049
34713,95682,2,50.000000,50.000000,0.000000,3.125000,15.625000,21.875000,25.000000,9.375000,...,9.027778,7.638889,6.250000,45.138889,6.944444,15.972222,4.166666,3.472222,22.222223,150.857143


### Autres colonnes

In [62]:
'''
"- **'proportion d'actifs'** (float, somme des proportions d'actifs) ou **'majorité d'actifs'** (booléen)\n",
"- **'csp majoritaire'** (str à one-hot encoder ou label encoder)\n",
"- **'tranche d'âge majoritaire'** (str à one-hot encoder ou label encoder)\n",
"- exclure les colonnes inutiles avec la stratégie **select k best**"
'''

# On supprime 'homme seul' et 'femme seule' (déjà inclus dans 'personne seule')
df = df.drop(columns=['Femme seule', 'Homme seul'])

# On transforme 'population_active' en un booléen 'grande_ville'
df['Grande ville'] = df['Population_active'].map(lambda x: True if x >= 10000 else False)
df = df.drop(columns=['Population_active'])


# On ajoute un booléen pour savoir si la commune possède une majorité de personnes actives
df['Total actifs'] = df['Agriculteurs'] + df['Artisans'] + df['Cadres'] + df['Intermédiaires'] + df['Employés'] + df['Ouvriers']
df['Majorité actifs'] = df['Total actifs'].map(lambda x: True if x > 50 else False)
df = df.drop(columns=['Total actifs'])
display(df)




,Code_INSEE,Résultat,Femmes,Hommes,Agriculteurs,Artisans,Cadres,Intermédiaires,Employés,Ouvriers,...,65-79 ans,80 ans et +,Mariés,Pacsés,Concubinage,Veufs,Divorcés,Célibataires,Grande ville,Majorité actifs
0,01001,2,47.184259,52.113958,0.704243,5.633941,10.563640,14.789096,15.493339,17.606067,...,18.364419,6.312769,51.219512,7.890961,16.499283,5.882353,3.156385,15.351506,False,True
1,01002,0,41.363637,50.555556,11.489899,2.297980,9.191919,13.787879,9.191919,9.191919,...,14.814815,8.333333,47.685185,11.111111,9.722222,3.240741,7.407407,20.833333,False,True
2,01004,2,52.881826,47.118174,0.127297,3.114317,7.806622,15.437689,17.760558,15.886919,...,16.198957,6.106620,40.957640,5.835323,9.955739,6.778822,8.566899,27.644693,True,True
3,01005,2,51.346262,47.882309,0.362612,6.556032,8.691036,15.289281,21.150485,13.928360,...,15.384787,5.562916,48.809515,7.840915,14.193995,4.623646,4.509098,20.022831,False,True
4,01006,2,39.860139,54.807692,0.000000,0.000000,0.000000,9.965035,14.947553,44.842658,...,25.000000,5.769231,49.038462,8.653846,10.576923,2.884615,9.615385,19.230769,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34710,95676,2,54.649893,44.135551,0.000000,5.257478,9.135020,24.653834,20.575269,6.438841,...,16.979097,4.528522,47.392614,4.639004,15.669851,5.102841,6.176331,21.019359,False,True
34711,95678,2,45.606723,51.307563,0.000000,7.838655,16.389916,20.665546,10.689076,5.700840,...,13.714286,7.142857,45.714286,10.142857,12.857143,5.000000,4.571429,21.714286,False,True
34712,95680,0,51.602206,48.397794,0.000000,2.962626,4.086016,11.996908,20.954176,17.632084,...,10.714654,3.277626,44.969973,1.178277,7.481139,4.148062,5.286505,36.832896,True,True
34713,95682,2,50.000000,50.000000,0.000000,3.125000,15.625000,21.875000,25.000000,9.375000,...,7.638889,6.250000,45.138889,6.944444,15.972222,4.166666,3.472222,22.222223,False,True


# Modélisation

### Split test-train

In [63]:
X = df.drop(columns=['Code_INSEE', 'Résultat'])
y = df['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

✓ Données préparées
  Train: (24300, 32)
  Test: (10415, 32)


### Fonctions

In [64]:
groups = ['gauche', 'centre', 'droite']


def train_model_and_detect_overfitting(model):
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)


    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    gap = abs(train_score - test_score)


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%}\n")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif train_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre\n")
    
    return y_pred_test, y_pred_train


def display_confusion_matrix(y, y_pred):
    matrix = confusion_matrix(y, y_pred, labels=groups)

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=[f'Prédit {g}' for g in groups],
                yticklabels=[f'Réel {g}' for g in groups])
    plt.title('Matrice de confusion', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


def display_metrics(y, y_pred):
    print("\n\nRapport de classification :")
    print(classification_report(y, y_pred, target_names=groups))

### Régression logistique + SMOTE

In [66]:
logistic_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])


logistic_smote.fit(X_train, y_train)
lrs_y_pred_test, lrs_y_pred_train = train_model_and_detect_overfitting(logistic_smote)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TEST\n')
display_metrics(y_test, lrs_y_pred_test)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, lrs_y_pred_train)



  Train : 58.52%
  Test  : 58.38%
  Gap   : 0.14%

⚠️ UNDERFITTING détecté !

---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TEST



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.27      0.56      0.36      1049
      centre       0.28      0.52      0.36      1524
      droite       0.88      0.60      0.71      7842

    accuracy                           0.58     10415
   macro avg       0.47      0.56      0.48     10415
weighted avg       0.73      0.58      0.63     10415


---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + SMOTE SUR LE TRAIN



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.28      0.58      0.37      2449
      centre       0.27      0.51      0.36      3555
      droite       0.88      0.60      0.71     18296

    accuracy                           0.59     24300
   macro avg       0.48      0.56      0.48     24300
weighted avg       0.73      

### Régression logistique + weighted

In [67]:
logistic_weighted = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
])


logistic_weighted.fit(X_train, y_train)
lrw_y_pred_test, lrw_y_pred_train = train_model_and_detect_overfitting(logistic_weighted)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TEST\n')
display_metrics(y_test, lrw_y_pred_test)


print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train, lrw_y_pred_train)

  Train : 59.13%
  Test  : 58.89%
  Gap   : 0.25%

⚠️ UNDERFITTING détecté !

---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TEST



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.27      0.55      0.37      1049
      centre       0.27      0.51      0.35      1524
      droite       0.88      0.61      0.72      7842

    accuracy                           0.59     10415
   macro avg       0.47      0.56      0.48     10415
weighted avg       0.73      0.59      0.63     10415


---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE + WEIGHTED SUR LE TRAIN



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.29      0.57      0.38      2449
      centre       0.27      0.51      0.35      3555
      droite       0.88      0.61      0.72     18296

    accuracy                           0.59     24300
   macro avg       0.48      0.56      0.48     24300
weighted avg       0.73

### XGBoost + Smote

In [68]:
xgb_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smote.fit(X_train, y_train)
xgbs_y_pred_test, xgbs_y_pred_train = train_model_and_detect_overfitting(xgb_smote)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST\n')
display_metrics(y_test, xgbs_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, xgbs_y_pred_train)

  Train : 75.47%
  Test  : 66.57%
  Gap   : 8.90%

✅ Bon équilibre


---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.29      0.48      0.36      1049
      centre       0.35      0.44      0.39      1524
      droite       0.85      0.74      0.79      7842

    accuracy                           0.67     10415
   macro avg       0.50      0.55      0.51     10415
weighted avg       0.72      0.67      0.69     10415


---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.45      0.73      0.56      2449
      centre       0.50      0.62      0.55      3555
      droite       0.90      0.78      0.84     18296

    accuracy                           0.75     24300
   macro avg       0.62      0.71      0.65     24300
weighted avg       0.80      0.75      0.77     24300



### XGBoost + SmoteEEN

In [69]:
xgb_smoteenn = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTEENN(random_state=42, smote=SMOTE(sampling_strategy='all', random_state=42))),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smoteenn.fit(X_train, y_train)
xgbse_y_pred_test, xgbse_y_pred_train = train_model_and_detect_overfitting(xgb_smoteenn)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST\n')
display_metrics(y_test, xgbse_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN\n')
display_metrics(y_train, xgbse_y_pred_train)

  Train : 53.87%
  Test  : 46.66%
  Gap   : 7.21%

⚠️ UNDERFITTING détecté !

---RÉSULTATS DE XGBOOST + SMOTE SUR LE TEST



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.22      0.62      0.33      1049
      centre       0.25      0.64      0.36      1524
      droite       0.91      0.41      0.57      7842

    accuracy                           0.47     10415
   macro avg       0.46      0.56      0.42     10415
weighted avg       0.75      0.47      0.51     10415


---RÉSULTATS DE XGBOOST + SMOTE SUR LE TRAIN



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.32      0.89      0.47      2449
      centre       0.32      0.82      0.46      3555
      droite       0.96      0.44      0.60     18296

    accuracy                           0.54     24300
   macro avg       0.53      0.72      0.51     24300
weighted avg       0.80      0.54      0.57     24300



### XGBoost + Smote Tomek

In [72]:
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks


xgb_smotetomek = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTETomek(tomek=TomekLinks(sampling_strategy='all'), random_state=42)),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


xgb_smotetomek.fit(X_train, y_train)
xgbst_y_pred_test, xgbst_y_pred_train = train_model_and_detect_overfitting(xgb_smotetomek)


print('\n---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TEST\n')
display_metrics(y_test, xgbst_y_pred_test)


print('\n---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TRAIN\n')
display_metrics(y_train, xgbst_y_pred_train)

  Train : 75.47%
  Test  : 66.56%
  Gap   : 8.91%

✅ Bon équilibre


---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TEST



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.29      0.47      0.36      1049
      centre       0.34      0.43      0.38      1524
      droite       0.85      0.74      0.79      7842

    accuracy                           0.67     10415
   macro avg       0.49      0.55      0.51     10415
weighted avg       0.72      0.67      0.69     10415


---RÉSULTATS DE XGBOOST + SMOTE TOMEK SUR LE TRAIN



Rapport de classification :
              precision    recall  f1-score   support

      gauche       0.45      0.73      0.56      2449
      centre       0.50      0.62      0.56      3555
      droite       0.90      0.78      0.84     18296

    accuracy                           0.75     24300
   macro avg       0.62      0.71      0.65     24300
weighted avg       0.80      0.75      0.77     24300

